In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp "/content/drive/MyDrive/Colab Notebooks/image_rec/coco_dataset/annotations/filtered_train2017.json" "/content"
!cp "/content/drive/MyDrive/Colab Notebooks/image_rec/coco_dataset/annotations/filtered_val2017.json" "/content"

将json格式的数据标注转换为yolo格式的数据标注

In [3]:
import json
import os
from tqdm import tqdm

def convert_to_yolo(json_path, output_label_dir):
    with open(json_path, 'r') as f:
        data = json.load(f)

    os.makedirs(output_label_dir, exist_ok=True)

    # 建立类别 ID 映射（0-27）
    cat_ids = sorted([cat['id'] for cat in data['categories']])
    id_map = {old_id: new_id for new_id, old_id in enumerate(cat_ids)}

    # 建立图片信息索引
    images = {img['id']: img for img in data['images']}

    # 写入 .txt 文件
    for ann in tqdm(data['annotations']):
        img = images[ann['image_id']]
        img_w, img_h = img['width'], img['height']

        # YOLO 格式: <class_id> <x_center> <y_center> <width> <height> (归一化)
        x, y, w, h = ann['bbox']
        x_center = (x + w / 2) / img_w
        y_center = (y + h / 2) / img_h
        nw = w / img_w
        nh = h / img_h

        cls_id = id_map[ann['category_id']]
        line = f"{cls_id} {x_center:.6f} {y_center:.6f} {nw:.6f} {nh:.6f}\n"

        txt_name = os.path.splitext(img['file_name'])[0] + ".txt"
        with open(os.path.join(output_label_dir, txt_name), 'a') as f:
            f.write(line)

# 执行转换
convert_to_yolo('filtered_train2017.json',  '/content/datasets/labels/train2017')
convert_to_yolo('filtered_val2017.json',  '/content/datasets/labels/val2017')

100%|██████████| 9617/9617 [00:00<00:00, 34399.74it/s]


/content/datasets/
    ├── images/
    │   ├── train/ (放入 train2017.zip 解压后的图)
    │   └── val/   (放入 val2017.zip 解压后的图)
    └── labels/
        ├── train/ (上一步转换生成的 .txt)
        └── val/   (上一步转换生成的 .txt)
    ——data.yaml

In [4]:
!cp "/content/drive/MyDrive/Colab Notebooks/image_rec/coco_dataset/filtered_train2017.zip" "/content"
!cp "/content/drive/MyDrive/Colab Notebooks/image_rec/coco_dataset/filtered_val2017.zip" "/content"


In [6]:
# 在 Colab 单元格执行
!unzip -q "/content/filtered_train2017.zip" -d "/content/datasets/images"
!unzip -q "/content/filtered_val2017.zip" -d "/content/datasets/images"

!cp "/content/drive/MyDrive/Colab Notebooks/image_rec/data.yaml" "/content/datasets/"

In [5]:
!git clone https://github.com/ultralytics/ultralytics.git
!pip install ultralytics

Cloning into 'ultralytics'...
remote: Enumerating objects: 89498, done.
remote: Counting objects: 100% (1415/1415), done.
remote: Compressing objects: 100% (434/434), done.
remote: Total 89498 (delta 1296), reused 983 (delta 981), pack-reused 88083 (from 3)
Receiving objects: 100% (89498/89498), 48.59 MiB | 16.40 MiB/s, done.
Resolving deltas: 100% (67433/67433), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 62.1 MB/s eta 0:00:00


In [1]:
import sys
import os

# 将你修改过的那个 ultralytics 文件夹的父目录加入系统路径
# 注意：路径要指向包含 ultralytics 文件夹的那个目录
sys.path.insert(0, '/content/ultralytics')

from ultralytics import YOLO
# 验证路径：运行下面这行，确保输出的是 /content/ultralytics/...
import ultralytics
print(ultralytics.__file__)

/content/ultralytics/ultralytics/__init__.py


如果上面格子的代码不生效

In [4]:
#!pip uninstall ultralytics -y

Found existing installation: ultralytics 8.4.21
Uninstalling ultralytics-8.4.21:
  Successfully uninstalled ultralytics-8.4.21


In [2]:
 # # 打印出 CBAM 类的实际文件路径
    # print("="*50)
    # print(f"当前脚本正在使用的 CBAM 类的源文件路径是: \n{inspect.getfile(CBAM)}")
    # print("="*50)
from ultralytics import YOLO
!mkdir -p /content/yolov8_object_detection
%cd /content/yolov8_object_detection
import torch
torch.use_deterministic_algorithms(False)

model = YOLO('/content/drive/MyDrive/Colab Notebooks/yolo_detection/yolov8_object_detection/yolov8.yaml')

    # ✅ 加入以下代码，打印出模型实际加载的 YAML 内容
    # print("\n" + "="*25 + " 模型实际加载的 YAML 内容 " + "="*25)
    # print(yaml.dump(model.yaml))
    # print("="*70 + "\n")

    # 2. Load the pre-trained weights (e.g., yolov8n.pt)
    # The .load() method intelligently transfers matching layers from the .pt file
model.load('yolov8n.pt')

    # 3. Start training
model.train(
        data='/content/datasets/data.yaml',
        epochs=200,
        imgsz=640,
        plots=True,
        patience=20,
        batch=-1,
    )

/content/yolov8_object_detection
WARNING ⚠️ no model scale passed. Assuming scale='n'.
Transferred 355/355 items from pretrained weights
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Cola

Process Process-1:
Traceback (most recent call last):


: 0% ──────────── 0/728  8.7s

  File "/usr/lib/python3.12/multiprocessing/process.py", line 317, in _bootstrap
    util._exit_function()


  File "/usr/lib/python3.12/multiprocessing/util.py", line 363, in _exit_function
    _run_finalizers()
  File "/usr/lib/python3.12/multiprocessing/util.py", line 303, in _run_finalizers
    finalizer()
Exception in thread Thread-5 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
Exception in thread Thread-4 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  

RuntimeError: DataLoader worker (pid 259204) exited unexpectedly with exit code 1. Details are lost due to multiprocessing. Rerunning with num_workers=0 may give better error trace.

In [3]:

# # 打印出 CBAM 类的实际文件路径
    # print("="*50)
    # print(f"当前脚本正在使用的 CBAM 类的源文件路径是: \n{inspect.getfile(CBAM)}")
    # print("="*50)
from ultralytics import YOLO
!mkdir -p /content/yolov8_object_detection
%cd /content/yolov8_object_detection

import torch
torch.use_deterministic_algorithms(False)

model = YOLO('/content/drive/MyDrive/Colab Notebooks/image_rec/yolov8n_cbam_dsconv.yaml')

    # ✅ 加入以下代码，打印出模型实际加载的 YAML 内容
    # print("\n" + "="*25 + " 模型实际加载的 YAML 内容 " + "="*25)
    # print(yaml.dump(model.yaml))
    # print("="*70 + "\n")

    # 2. Load the pre-trained weights (e.g., yolov8n.pt)
    # The .load() method intelligently transfers matching layers from the .pt file
model.load('yolov8n.pt')
model.train(
    data='/content/datasets/data.yaml',
    epochs=200,
    imgsz=640,
    batch=64,
    workers=8,
    patience=30,
    plots=False,
)
%cd /content/ultralytics/runs/detect/
!zip -r "/content/drive/MyDrive/Colab Notebooks/image_rec/result1.zip" train

/content/yolov8_object_detection
Transferred 102/481 items from pretrained weights
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Colab Notebooks/image_rec/yolov8n_cbam_dsconv.yaml, moment

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/200      16.5G      2.881      4.516      3.391        295        640: 100% ━━━━━━━━━━━━ 990/990 2.3it/s 7:13
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 1.2s/it 25.8s
                   all       2714       9617      0.241       0.15     0.0716     0.0279

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/200      18.8G      1.874      3.121      2.068        235        640: 100% ━━━━━━━━━━━━ 990/990 2.4it/s 6:56
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 2.3it/s 9.4s
                   all       2714       9617      0.239      0.224      0.172     0.0912

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     55/200      18.8G      1.241      1.563      1.305        437        640: 95% ━━━━━━━━━━━─ 937/990 17.1it/s 6:31<3.1s


Exception ignored in: <generator object TQDM.__iter__ at 0x78b7843e7780>
Traceback (most recent call last):
  File "/content/ultralytics/ultralytics/utils/tqdm.py", line 354, in __iter__
KeyboardInterrupt: 
Process Process-39:
Process Process-33:
Process Process-24:
Traceback (most recent call last):
Process Process-27:
Process Process-25:
Process Process-34:
Traceback (most recent call last):
Process Process-23:
Traceback (most recent call last):
Process Process-35:
Process Process-36:
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 317, in _bootstrap
    util._exit_function()
Traceback (most recent call last):
Traceback (most recent call last):
Process Process-22:
Process Process-32:
Process Process-29:
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/util.py", line 363, in _exit_function
    _run_finalizers()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 317, in _bootstrap
    util._exit_f

KeyboardInterrupt: 

  File "/usr/lib/python3.12/multiprocessing/process.py", line 317, in _bootstrap
    util._exit_function()
  File "/usr/lib/python3.12/multiprocessing/util.py", line 363, in _exit_function
    _run_finalizers()
  File "/usr/lib/python3.12/multiprocessing/util.py", line 227, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 317, in _bootstrap
    util._exit_function()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 317, in _bootstrap
    util._exit_function()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 317, in _bootstrap
    util._exit_function()
  File "/usr/lib/python3.12/multiprocessing/util.py", line 227, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/util.py", line 303, in _run_finalizers
    finalizer()
 

继续训练的代码，epoch设定要大于当前训练到的epoch数（中断执行后才需要继续训练）

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/yolov8_object_detection/runs/detect/train/weights/last.pt")

model.train(
    data="/content/datasets/data.yaml",
    epochs=200,
    batch=64,
    imgsz=640,
    patience=30,
    plots=False,
    resume=True,
)

In [3]:
!rm -rf /content/ultralytics/runs/detect
#删除所有训练数据，重新开始

In [6]:
#!ls -lh filtered_train2017_fixed.zip

-rw------- 1 root root 679M Mar 13 06:29 filtered_train2017_fixed.zip


In [1]:
%cd /content/ultralytics/runs/detect/

/content/ultralytics/runs/detect


In [3]:
!zip -r "/content/drive/MyDrive/Colab Notebooks/image_rec/result2.zip" train2

  adding: train2/ (stored 0%)
  adding: train2/args.yaml (deflated 52%)
  adding: train2/results.csv (deflated 69%)
  adding: train2/weights/ (stored 0%)
  adding: train2/weights/best.pt (deflated 9%)
  adding: train2/weights/last.pt (deflated 9%)
